In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from numpy.linalg import norm
import os
import pandas as pd

from scipy.stats import skew
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

In [ ]:
input_directory  = '/eos/home-i00/d/dhnaik/C2V_event_training_data'
with open('/eos/home-i00/d/dhnaik/C2V_event_training_data/meta_dict.json') as f:
    meta_dict = json.load(f)
feature_dict = meta_dict['input_vars']
class_labels = meta_dict['class_labels']

In [ ]:
print(list(feature_dict.keys()))

In [ ]:
print(f'class labels: {class_labels}')

In [ ]:
X_features = np.load(input_directory+'/train/X_features.npy')
y_labels   = np.load(input_directory+'/train/y_label.npy')

In [ ]:
def preprocess(X,y,create_and_slice_features, normalise=False, test_size=0.3):
    
    X = create_and_slice_features(X)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y, shuffle=True)
    
    if normalise:
    
        train_mask = (X_train[..., 0] != 0)[..., np.newaxis] 
        test_mask = (X_test[..., 0] != 0)[..., np.newaxis]
        
        mean = np.mean(X_train, axis=(0, 1), keepdims=True)
        std = np.std(X_train, axis=(0, 1), keepdims=True)
        
        std = np.where(std == 0, 1e-7, std)
    
        # Normalize
        X_train = (X_train - mean) / std
        X_test = (X_test - mean) / std
        
        # RE-MASK: Force the padded particles back to exactly 0.0
        X_train = X_train * train_mask
        X_test = X_test * test_mask

        
    return X_train, X_test, y_train, y_test

In [ ]:
def just_feature_vector(X_cand,features=[],features_list=[]):
    selected_features = [features_list[feature] for feature in features ]
    X = X_cand[:,selected_features]
    
    return X

In [ ]:
jet_feature_list = ['L1T_JetPuppiAK4_PT','L1T_JetPuppiAK4_Eta','L1T_JetPuppiAK4_Phi']
muon_feature_list = ['L1T_MuonTight_PT','L1T_MuonTight_Eta','L1T_MuonTight_Phi']
electron_feature_list = ['L1T_Electron_PT','L1T_Electron_Eta','L1T_Electron_Phi']
met_feature_list = ['L1T_PUPPIMET_MET','L1T_PUPPIMET_Eta','L1T_PUPPIMET_Phi']

max_number_of_jets = 10
max_number_of_muons = 4
max_number_of_electrons = 4

top_x_jets = [feature + str(i) for i in range(max_number_of_jets) for feature in jet_feature_list ]
top_x_muons = [feature + str(i) for i in range(max_number_of_muons) for feature in muon_feature_list]
top_x_electrons = [feature + str(i) for i in range(max_number_of_electrons) for feature in electron_feature_list]
all_columns = top_x_jets + top_x_muons + top_x_electrons + met_feature_list

In [ ]:
X_train, X_test, y_train, y_test = preprocess(X_features,y_labels, 
                                              lambda X_features: just_feature_vector(X_features,features=all_columns,features_list=feature_dict),
                                              normalise=False)

In [ ]:
model_input_shape = X_train.shape[1:]
model_output_shape = len(class_labels.keys())
print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")
print(f'Feature Names: {all_columns}')
print(f'\nFeature Names: {list(feature_dict.keys())}')

## `yggdrasil gradient boosted decision tree`

In [ ]:
import ydf

In [ ]:
event_train_ds = {f'{all_columns[i]}' : X_train[:,i] for i in range(X_train.shape[1])}
event_train_ds['label'] = y_train.astype(int)

event_test_ds = {f'{all_columns[i]}' : X_test[:,i] for i in range(X_test.shape[1])}
event_test_ds['label'] = y_test.astype(int)

event_train_ds = pd.DataFrame(event_train_ds)
event_test_ds = pd.DataFrame(event_test_ds)

In [ ]:
event_train_ds

In [ ]:
event_train_subset = pd.concat([
    grp.sample(frac=0.3, random_state=42)
    for _, grp in event_train_ds.groupby('label')
]).reset_index(drop=True)

event_test_subset = pd.concat([
    grp.sample(frac=0.3, random_state=42)
    for _, grp in event_test_ds.groupby('label')
]).reset_index(drop=True)

In [ ]:
event_classifier_model = ydf.GradientBoostedTreesLearner(
    label='label',
    num_trees=500,
    shrinkage=0.08,
    subsample=0.8,
    min_examples=100,
    max_depth=8,
    growing_strategy='LOCAL',
    early_stopping='LOSS_INCREASE',
    early_stopping_num_trees_look_ahead=50,
    validation_ratio=0.1,
    l2_regularization=0.2,
    num_threads=64,
).train(event_train_ds, verbose=1)

In [ ]:
!pwd

In [ ]:
prediction = event_classifier_model.predict(event_test_ds)
prediction_2col = np.stack([1 - prediction, prediction], axis=1)  # (N, 2)
np.save('/eos/home-i00/d/dhnaik/SDT/test_outputs/EVENT_C2V/HH_4b_bdt_probs.npy', prediction_2col)

In [ ]:
prediction_2col

In [ ]:
evaluation = event_classifier_model.evaluate(event_test_ds, weighted=False)

In [ ]:
evaluation

In [ ]:
event_classifier_model.describe()